
# SecureSense — Complete Research Evidence Collection Notebook

**Purpose:** collect, verify, reproduce, and export the experimental evidence needed for the SecureSense research paper.

### Important
- **This notebook does NOT retrain any model.**
- It loads the saved DNN artifacts and the exact train/test Parquet files used by the final experiment.
- It keeps the **primary stratified DNN evaluation** and the **source-disjoint/unseen-source DNN evaluation** separate.
- It never replaces an actual result with a better-looking result.
- If an artifact is missing, the notebook reports the missing evidence instead of inventing it.

### Main evidence produced
1. Environment and reproducibility record
2. Dataset shape, labels, class distribution and numerical sanity checks
3. Exact 69-feature verification
4. Saved-model architecture and parameter count
5. Saved threshold/scaler/model verification
6. Primary DNN metrics from saved experiment output
7. Source-disjoint/unseen-source DNN metrics
8. Independent re-evaluation of the saved unseen-source model
9. Validation threshold sweep (0.10–0.90)
10. Threshold-vs-Precision/Recall/F1/FPR/FNR analysis
11. Confusion matrix
12. ROC curve and Precision–Recall curve
13. Class-support table
14. Inference-time benchmark
15. Optional historical Random Forest comparison (kept separate from the final DNN)
16. Export of all evidence tables and figures to `results/research_evidence`


In [ ]:

# ============================================================
# 0. CONFIGURATION — EDIT ONLY BASE_DIR IF NEEDED
# ============================================================

from pathlib import Path
import os
import sys
import json
import pickle
import time
import platform
import subprocess
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

warnings.filterwarnings("ignore")

# Exact project root found in the existing SecureSense training artifacts.
BASE_DIR = Path(r"W:\FinalYearProject\SecureSense_Project")

TRAIN_PATH = BASE_DIR / "notebooks" / "data" / "final_group_splits" / "train.parquet"
TEST_PATH  = BASE_DIR / "notebooks" / "data" / "final_group_splits" / "test.parquet"

FEATURE_PATH = BASE_DIR / "models" / "final_feature_names.pkl"

# Final source-disjoint / unseen-source DNN
FINAL_MODEL_PATH     = BASE_DIR / "models" / "final_dnn.keras"
FINAL_SCALER_PATH    = BASE_DIR / "models" / "final_dnn_scaler.pkl"
FINAL_THRESHOLD_PATH = BASE_DIR / "models" / "final_dnn_threshold.pkl"
FINAL_METRICS_PATH   = BASE_DIR / "results" / "final_dnn_metrics.csv"

# Primary stratified held-out DNN
PRIMARY_MODEL_PATH     = BASE_DIR / "models" / "primary_dnn.keras"
PRIMARY_SCALER_PATH    = BASE_DIR / "models" / "primary_dnn_scaler.pkl"
PRIMARY_THRESHOLD_PATH = BASE_DIR / "models" / "primary_dnn_threshold.pkl"
PRIMARY_METRICS_PATH   = BASE_DIR / "results" / "final_primary_dnn" / "primary_dnn_test_metrics.csv"
PRIMARY_CONFIG_PATH    = BASE_DIR / "results" / "final_primary_dnn" / "primary_dnn_configuration.json"

# All newly generated research evidence goes here.
EVIDENCE_DIR = BASE_DIR / "results" / "research_evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
VALIDATION_SIZE = 0.10
BATCH_SIZE = 8192
CHUNK_SIZE = 100_000

print("SecureSense evidence notebook")
print("Project:", BASE_DIR)
print("Evidence output:", EVIDENCE_DIR)


In [ ]:

# ============================================================
# 1. ENVIRONMENT / REPRODUCIBILITY RECORD
# ============================================================

def package_version(name):
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "unknown")
    except Exception:
        return "not installed"

env = {
    "python": sys.version,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "sklearn": package_version("sklearn"),
    "matplotlib": package_version("matplotlib"),
    "tensorflow": package_version("tensorflow"),
}

env_df = pd.DataFrame(list(env.items()), columns=["Item", "Value"])
display(env_df)

env_df.to_csv(EVIDENCE_DIR / "environment.csv", index=False)

print("\nTensorFlow GPU devices:")
try:
    import tensorflow as tf
    print(tf.config.list_physical_devices("GPU"))
except Exception as e:
    print("TensorFlow unavailable:", e)


In [ ]:

# ============================================================
# 2. ARTIFACT INVENTORY — DO NOT PROCEED SILENTLY IF MISSING
# ============================================================

artifact_paths = {
    "train_parquet": TRAIN_PATH,
    "test_parquet": TEST_PATH,
    "feature_list": FEATURE_PATH,
    "final_dnn_model": FINAL_MODEL_PATH,
    "final_dnn_scaler": FINAL_SCALER_PATH,
    "final_dnn_threshold": FINAL_THRESHOLD_PATH,
    "final_dnn_metrics": FINAL_METRICS_PATH,
    "primary_dnn_model": PRIMARY_MODEL_PATH,
    "primary_dnn_scaler": PRIMARY_SCALER_PATH,
    "primary_dnn_threshold": PRIMARY_THRESHOLD_PATH,
    "primary_dnn_metrics": PRIMARY_METRICS_PATH,
    "primary_dnn_configuration": PRIMARY_CONFIG_PATH,
}

artifact_rows = []
for name, path in artifact_paths.items():
    artifact_rows.append({
        "artifact": name,
        "path": str(path),
        "exists": path.exists(),
        "size_MB": round(path.stat().st_size / (1024**2), 3) if path.exists() and path.is_file() else np.nan
    })

artifact_df = pd.DataFrame(artifact_rows)
display(artifact_df)

artifact_df.to_csv(EVIDENCE_DIR / "artifact_inventory.csv", index=False)

missing = artifact_df.loc[~artifact_df["exists"], "artifact"].tolist()
if missing:
    print("\nWARNING — missing artifacts:")
    print(*missing, sep="\n")
    print("\nThe notebook will skip sections that require unavailable artifacts.")
else:
    print("\nAll expected research artifacts are present.")


In [ ]:

# ============================================================
# 3. LOAD EXACT 69-FEATURE LIST
# ============================================================

feature_names = None

if FEATURE_PATH.exists():
    with open(FEATURE_PATH, "rb") as f:
        feature_names = list(pickle.load(f))

    print("Number of model features:", len(feature_names))
    print("69-feature check:", "PASSED" if len(feature_names) == 69 else "FAILED")

    feature_df = pd.DataFrame({
        "index": range(1, len(feature_names) + 1),
        "feature": feature_names
    })
    display(feature_df)

    feature_df.to_csv(EVIDENCE_DIR / "final_69_features.csv", index=False)
else:
    print("Feature-list artifact not found.")


In [ ]:

# ============================================================
# 4. LOAD DATASET SPLITS + DATASET AUDIT
# ============================================================

train_df = None
test_df = None

if TRAIN_PATH.exists() and TEST_PATH.exists():
    print("Loading train/test Parquet files...")
    train_df = pd.read_parquet(TRAIN_PATH)
    test_df = pd.read_parquet(TEST_PATH)

    print("Train shape:", train_df.shape)
    print("Test shape :", test_df.shape)

    if feature_names is not None:
        missing_train = [f for f in feature_names if f not in train_df.columns]
        missing_test = [f for f in feature_names if f not in test_df.columns]

        print("\nFeature-schema check:")
        print("Missing train features:", missing_train)
        print("Missing test features :", missing_test)

    def convert_labels(series):
        labels = series.astype(str).str.strip().str.upper()
        return np.where(labels == "BENIGN", 0, 1).astype(np.int32)

    y_train_full = convert_labels(train_df["Label"])
    y_test = convert_labels(test_df["Label"])

    dataset_audit = pd.DataFrame([
        {
            "split": "train",
            "rows": len(train_df),
            "columns": train_df.shape[1],
            "benign": int((y_train_full == 0).sum()),
            "attack": int((y_train_full == 1).sum()),
            "attack_rate_percent": round(float((y_train_full == 1).mean() * 100), 4),
            "duplicate_rows": int(train_df.duplicated().sum()),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "columns": test_df.shape[1],
            "benign": int((y_test == 0).sum()),
            "attack": int((y_test == 1).sum()),
            "attack_rate_percent": round(float((y_test == 1).mean() * 100), 4),
            "duplicate_rows": int(test_df.duplicated().sum()),
        },
    ])

    display(dataset_audit)
    dataset_audit.to_csv(EVIDENCE_DIR / "dataset_audit.csv", index=False)

    # Label counts
    label_df = pd.DataFrame({
        "split": ["train", "test"],
        "BENIGN": [int((y_train_full == 0).sum()), int((y_test == 0).sum())],
        "ATTACK": [int((y_train_full == 1).sum()), int((y_test == 1).sum())],
    })
    display(label_df)
    label_df.to_csv(EVIDENCE_DIR / "class_distribution.csv", index=False)

    # Numerical sanity check on exact 69 features
    X_train_raw = train_df[feature_names].astype(np.float32)
    X_test_raw = test_df[feature_names].astype(np.float32)

    numerical_audit = pd.DataFrame([
        {
            "split": "train",
            "rows": len(X_train_raw),
            "features": X_train_raw.shape[1],
            "nan_values": int(X_train_raw.isna().sum().sum()),
            "positive_infinity": int(np.isposinf(X_train_raw.to_numpy()).sum()),
            "negative_infinity": int(np.isneginf(X_train_raw.to_numpy()).sum()),
        },
        {
            "split": "test",
            "rows": len(X_test_raw),
            "features": X_test_raw.shape[1],
            "nan_values": int(X_test_raw.isna().sum().sum()),
            "positive_infinity": int(np.isposinf(X_test_raw.to_numpy()).sum()),
            "negative_infinity": int(np.isneginf(X_test_raw.to_numpy()).sum()),
        },
    ])
    display(numerical_audit)
    numerical_audit.to_csv(EVIDENCE_DIR / "numerical_sanity.csv", index=False)
else:
    print("Dataset files not available; dataset audit skipped.")


In [ ]:

# ============================================================
# 5. MODEL / SCALER / THRESHOLD INSPECTION
# ============================================================

import tensorflow as tf

models = {}

for name, path in [
    ("final_dnn", FINAL_MODEL_PATH),
    ("primary_dnn", PRIMARY_MODEL_PATH),
]:
    if path.exists():
        print(f"\nLoading {name}: {path}")
        models[name] = tf.keras.models.load_model(path, compile=False)
        models[name].summary()

        summary_lines = []
        models[name].summary(print_fn=summary_lines.append)

        (EVIDENCE_DIR / f"{name}_architecture.txt").write_text(
            "\n".join(summary_lines),
            encoding="utf-8"
        )

        print("Parameter count:", models[name].count_params())

for name, path in [
    ("final_dnn_scaler", FINAL_SCALER_PATH),
    ("primary_dnn_scaler", PRIMARY_SCALER_PATH),
]:
    if path.exists():
        with open(path, "rb") as f:
            obj = pickle.load(f)
        print(f"{name}: loaded; type = {type(obj).__name__}")

thresholds_loaded = {}
for name, path in [
    ("final_dnn", FINAL_THRESHOLD_PATH),
    ("primary_dnn", PRIMARY_THRESHOLD_PATH),
]:
    if path.exists():
        with open(path, "rb") as f:
            thresholds_loaded[name] = float(pickle.load(f))
        print(f"{name} threshold:", thresholds_loaded[name])

threshold_df = pd.DataFrame(
    [{"model": k, "saved_threshold": v} for k, v in thresholds_loaded.items()]
)
display(threshold_df)
threshold_df.to_csv(EVIDENCE_DIR / "saved_thresholds.csv", index=False)


In [ ]:

# ============================================================
# 6. PRIMARY DNN — LOAD SAVED TEST METRICS
# ============================================================

if PRIMARY_METRICS_PATH.exists():
    primary_metrics = pd.read_csv(PRIMARY_METRICS_PATH)
    print("Primary DNN saved metrics:")
    display(primary_metrics)

    primary_metrics.to_csv(
        EVIDENCE_DIR / "primary_dnn_saved_metrics.csv", index=False
    )

if PRIMARY_CONFIG_PATH.exists():
    print("\nPrimary DNN configuration:")
    try:
        primary_config = json.loads(PRIMARY_CONFIG_PATH.read_text(encoding="utf-8"))
        display(pd.DataFrame(
            [{"parameter": k, "value": v} for k, v in primary_config.items()]
        ))
    except Exception as e:
        print("Could not parse configuration:", e)


In [ ]:

# ============================================================
# 7. RECREATE THE EXACT FINAL-DNN VALIDATION SPLIT
#    (NO RETRAINING)
# ============================================================

final_validation_ready = False

if train_df is not None and "final_dnn" in models and FINAL_SCALER_PATH.exists():
    with open(FINAL_SCALER_PATH, "rb") as f:
        final_scaler = pickle.load(f)

    # The original final DNN script used:
    # train_test_split(..., test_size=0.10, random_state=42, stratify=y)
    indices = np.arange(len(y_train_full))

    train_indices, val_indices = train_test_split(
        indices,
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_STATE,
        stratify=y_train_full,
    )

    X_train_raw_final = train_df.iloc[train_indices][feature_names].astype(np.float32).to_numpy()
    X_val_raw_final   = train_df.iloc[val_indices][feature_names].astype(np.float32).to_numpy()
    y_val_final       = y_train_full[val_indices]

    # The saved scaler was fit on the training portion.
    X_val_final = final_scaler.transform(X_val_raw_final).astype(np.float32)

    print("Validation samples:", len(y_val_final))
    print("Validation BENIGN :", int((y_val_final == 0).sum()))
    print("Validation ATTACK :", int((y_val_final == 1).sum()))

    final_validation_ready = True

    del X_train_raw_final
    del X_val_raw_final
    del train_indices
    del val_indices
else:
    print("Final validation reconstruction unavailable.")


In [ ]:

# ============================================================
# 8. THRESHOLD SWEEP ON VALIDATION
# ============================================================

threshold_results = None
val_probabilities = None

if final_validation_ready:
    final_model = models["final_dnn"]

    print("Generating validation probabilities...")
    val_probabilities = final_model.predict(
        X_val_final,
        batch_size=BATCH_SIZE,
        verbose=1
    ).ravel()

    thresholds = np.arange(0.10, 0.91, 0.01)
    rows = []

    for threshold in thresholds:
        pred = (val_probabilities >= threshold).astype(np.int32)
        tn, fp, fn, tp = confusion_matrix(
            y_val_final, pred, labels=[0, 1]
        ).ravel()

        rows.append({
            "threshold": round(float(threshold), 2),
            "accuracy": accuracy_score(y_val_final, pred),
            "precision": precision_score(y_val_final, pred, zero_division=0),
            "recall": recall_score(y_val_final, pred, zero_division=0),
            "f1": f1_score(y_val_final, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_val_final, pred),
            "fpr": fp / (fp + tn) if (fp + tn) else np.nan,
            "fnr": fn / (fn + tp) if (fn + tp) else np.nan,
            "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        })

    threshold_results = pd.DataFrame(rows)
    display(threshold_results)

    # Verify the original threshold-selection rule: maximize validation accuracy.
    best_row = threshold_results.loc[threshold_results["accuracy"].idxmax()]
    print("\nThreshold selected by maximum validation accuracy:")
    display(best_row.to_frame("value"))

    saved_threshold = thresholds_loaded.get("final_dnn", np.nan)
    print("Saved threshold:", saved_threshold)
    print("Reconstructed best threshold:", best_row["threshold"])

    threshold_results.to_csv(
        EVIDENCE_DIR / "threshold_sweep_validation.csv", index=False
    )
else:
    print("Threshold sweep skipped.")


In [ ]:

# ============================================================
# 9. THRESHOLD TRADE-OFF FIGURES
# ============================================================

if threshold_results is not None:
    plt.figure(figsize=(10, 6))
    plt.plot(threshold_results["threshold"], threshold_results["precision"], label="Precision")
    plt.plot(threshold_results["threshold"], threshold_results["recall"], label="Recall")
    plt.plot(threshold_results["threshold"], threshold_results["f1"], label="F1-score")
    if not np.isnan(saved_threshold):
        plt.axvline(saved_threshold, linestyle="--", label=f"Saved threshold = {saved_threshold:.2f}")
    plt.xlabel("Decision threshold")
    plt.ylabel("Score")
    plt.title("SecureSense DNN — Threshold vs Precision / Recall / F1")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(EVIDENCE_DIR / "threshold_precision_recall_f1.png", dpi=300)
    plt.show()

    plt.figure(figsize=(10, 6))
    plt.plot(threshold_results["threshold"], threshold_results["fpr"], label="False Positive Rate")
    plt.plot(threshold_results["threshold"], threshold_results["fnr"], label="False Negative Rate")
    if not np.isnan(saved_threshold):
        plt.axvline(saved_threshold, linestyle="--", label=f"Saved threshold = {saved_threshold:.2f}")
    plt.xlabel("Decision threshold")
    plt.ylabel("Rate")
    plt.title("SecureSense DNN — Threshold vs FPR / FNR")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(EVIDENCE_DIR / "threshold_fpr_fnr.png", dpi=300)
    plt.show()


In [ ]:

# ============================================================
# 10. RECOMPUTE FINAL UNSEEN-SOURCE TEST METRICS
# ============================================================

final_test_results = None
final_test_probabilities = None
final_test_predictions = None

if test_df is not None and "final_dnn" in models and FINAL_SCALER_PATH.exists():
    with open(FINAL_SCALER_PATH, "rb") as f:
        final_scaler = pickle.load(f)

    X_test_final_raw = test_df[feature_names].astype(np.float32).to_numpy()
    X_test_final = final_scaler.transform(X_test_final_raw).astype(np.float32)

    final_test_model = models["final_dnn"]
    saved_threshold = thresholds_loaded.get("final_dnn", 0.50)

    print("Generating final unseen-source test probabilities...")
    start = time.perf_counter()
    final_test_probabilities = final_test_model.predict(
        X_test_final,
        batch_size=BATCH_SIZE,
        verbose=1
    ).ravel()
    prediction_seconds = time.perf_counter() - start

    final_test_predictions = (
        final_test_probabilities >= saved_threshold
    ).astype(np.int32)

    tn, fp, fn, tp = confusion_matrix(
        y_test, final_test_predictions, labels=[0, 1]
    ).ravel()

    final_test_results = pd.DataFrame([{
        "Evaluation": "Final source-disjoint / unseen-source test",
        "Threshold": saved_threshold,
        "Accuracy": accuracy_score(y_test, final_test_predictions),
        "Precision": precision_score(y_test, final_test_predictions, zero_division=0),
        "Recall": recall_score(y_test, final_test_predictions, zero_division=0),
        "F1": f1_score(y_test, final_test_predictions, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y_test, final_test_predictions),
        "ROC_AUC": roc_auc_score(y_test, final_test_probabilities),
        "PR_AUC": average_precision_score(y_test, final_test_probabilities),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "Specificity": tn / (tn + fp),
        "FPR": fp / (tn + fp),
        "FNR": fn / (fn + tp),
        "Prediction_Time_Seconds": prediction_seconds,
        "Average_ms_per_sample": (prediction_seconds / len(y_test)) * 1000,
    }])

    display(final_test_results)

    final_test_results.to_csv(
        EVIDENCE_DIR / "final_unseen_source_recomputed_metrics.csv",
        index=False
    )

    cm_df = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Actual BENIGN", "Actual ATTACK"],
        columns=["Predicted BENIGN", "Predicted ATTACK"]
    )
    display(cm_df)
    cm_df.to_csv(EVIDENCE_DIR / "final_unseen_source_confusion_matrix.csv")
else:
    print("Final unseen-source test re-evaluation skipped.")


In [ ]:

# ============================================================
# 11. FINAL CONFUSION MATRIX FIGURE
# ============================================================

if final_test_results is not None:
    cm = np.array([
        [int(final_test_results.loc[0, "TN"]), int(final_test_results.loc[0, "FP"])],
        [int(final_test_results.loc[0, "FN"]), int(final_test_results.loc[0, "TP"])]
    ])

    plt.figure(figsize=(7, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title("SecureSense DNN — Final Unseen-Source Confusion Matrix")
    plt.colorbar()
    plt.xticks([0, 1], ["Predicted BENIGN", "Predicted ATTACK"])
    plt.yticks([0, 1], ["Actual BENIGN", "Actual ATTACK"])
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center")

    plt.tight_layout()
    plt.savefig(EVIDENCE_DIR / "final_unseen_source_confusion_matrix.png", dpi=300)
    plt.show()


In [ ]:

# ============================================================
# 12. ROC + PRECISION–RECALL CURVES
# ============================================================

if final_test_probabilities is not None:
    fpr, tpr, _ = roc_curve(y_test, final_test_probabilities)
    precision_curve, recall_curve, _ = precision_recall_curve(
        y_test, final_test_probabilities
    )

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(
        f"SecureSense DNN — ROC Curve "
        f"(AUC = {roc_auc_score(y_test, final_test_probabilities):.4f})"
    )
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(EVIDENCE_DIR / "roc_curve.png", dpi=300)
    plt.show()

    plt.figure(figsize=(8, 6))
    plt.plot(recall_curve, precision_curve)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(
        f"SecureSense DNN — Precision–Recall Curve "
        f"(AP = {average_precision_score(y_test, final_test_probabilities):.4f})"
    )
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(EVIDENCE_DIR / "precision_recall_curve.png", dpi=300)
    plt.show()


In [ ]:

# ============================================================
# 13. FINAL CLASSIFICATION REPORT
# ============================================================

if final_test_predictions is not None:
    report_dict = classification_report(
        y_test,
        final_test_predictions,
        target_names=["BENIGN", "ATTACK"],
        output_dict=True,
        zero_division=0
    )

    report_df = pd.DataFrame(report_dict).T
    display(report_df)

    report_df.to_csv(
        EVIDENCE_DIR / "final_unseen_source_classification_report.csv"
    )


In [ ]:

# ============================================================
# 14. OPTIONAL ATTACK-TYPE / SOURCE ANALYSIS
# ============================================================

# This section does NOT assume a specific CICIDS2017 schema.
# It only runs if useful source/attack columns are actually present.

candidate_columns = [
    c for c in test_df.columns
    if any(token in c.lower() for token in [
        "label", "attack", "source", "group", "day", "host", "scenario"
    ])
]

print("Potential analysis columns found:")
print(candidate_columns)

for col in candidate_columns:
    if col == "Label":
        continue

    unique_count = test_df[col].nunique(dropna=False)
    print(f"\n{col}: {unique_count} unique values")
    if unique_count <= 30:
        display(test_df[col].value_counts(dropna=False).head(30).to_frame("count"))


In [ ]:

# ============================================================
# 15. INFERENCE BENCHMARK
# ============================================================

# Measures prediction time only; it does not claim end-to-end real-time performance.
# Use a fixed subset so this section is fast and reproducible.

if "final_dnn" in models and test_df is not None:
    benchmark_n = min(10_000, len(test_df))
    X_bench = X_test_final[:benchmark_n]

    # Warm-up
    _ = models["final_dnn"].predict(X_bench[:min(256, benchmark_n)], verbose=0)

    timings = []
    for _ in range(5):
        start = time.perf_counter()
        _ = models["final_dnn"].predict(X_bench, batch_size=BATCH_SIZE, verbose=0)
        timings.append(time.perf_counter() - start)

    benchmark_df = pd.DataFrame([{
        "samples": benchmark_n,
        "runs": len(timings),
        "mean_seconds": np.mean(timings),
        "std_seconds": np.std(timings),
        "mean_ms_per_sample": np.mean(timings) / benchmark_n * 1000,
        "samples_per_second": benchmark_n / np.mean(timings),
    }])

    display(benchmark_df)
    benchmark_df.to_csv(
        EVIDENCE_DIR / "inference_benchmark_10000.csv", index=False
    )


In [ ]:

# ============================================================
# 16. HISTORICAL RANDOM FOREST COMPARISON — OPTIONAL
# ============================================================

# This is retained only as historical/benchmark evidence if the old
# final_dnn_metrics.csv contains the comparison. It is NOT treated
# as the current deployed model.

if FINAL_METRICS_PATH.exists():
    try:
        saved_final_metrics = pd.read_csv(FINAL_METRICS_PATH)
        display(saved_final_metrics)
        saved_final_metrics.to_csv(
            EVIDENCE_DIR / "historical_saved_model_comparison.csv",
            index=False
        )
    except Exception as e:
        print("Could not read saved comparison:", e)
else:
    print("No historical comparison CSV found.")


In [ ]:

# ============================================================
# 17. CONSOLIDATED EVIDENCE SUMMARY
# ============================================================

summary_rows = []

if final_test_results is not None:
    r = final_test_results.iloc[0]
    summary_rows.extend([
        ["Final unseen-source Accuracy", r["Accuracy"]],
        ["Final unseen-source Precision", r["Precision"]],
        ["Final unseen-source Recall", r["Recall"]],
        ["Final unseen-source F1", r["F1"]],
        ["Final unseen-source ROC-AUC", r["ROC_AUC"]],
        ["Final unseen-source PR-AUC", r["PR_AUC"]],
        ["Final unseen-source Balanced Accuracy", r["Balanced_Accuracy"]],
        ["Final unseen-source Threshold", r["Threshold"]],
        ["Final unseen-source TN", r["TN"]],
        ["Final unseen-source FP", r["FP"]],
        ["Final unseen-source FN", r["FN"]],
        ["Final unseen-source TP", r["TP"]],
        ["Final unseen-source FPR", r["FPR"]],
        ["Final unseen-source FNR", r["FNR"]],
    ])

if feature_names is not None:
    summary_rows.append(["Number of model features", len(feature_names)])

if "final_dnn" in models:
    summary_rows.append(["Final DNN parameter count", models["final_dnn"].count_params()])

if threshold_results is not None:
    best = threshold_results.loc[threshold_results["accuracy"].idxmax()]
    summary_rows.extend([
        ["Reconstructed best validation threshold", best["threshold"]],
        ["Best validation accuracy", best["accuracy"]],
    ])

summary_df = pd.DataFrame(summary_rows, columns=["Evidence Item", "Value"])
display(summary_df)
summary_df.to_csv(EVIDENCE_DIR / "evidence_summary.csv", index=False)


In [ ]:

# ============================================================
# 18. RESEARCH-EVIDENCE MANIFEST
# ============================================================

manifest = {
    "project": "SecureSense",
    "purpose": "Research evidence collection; no model retraining",
    "primary_evaluation": "Stratified held-out DNN evaluation",
    "secondary_evaluation": "Source-disjoint / unseen-source DNN evaluation",
    "random_state": RANDOM_STATE,
    "validation_size": VALIDATION_SIZE,
    "model_features": len(feature_names) if feature_names is not None else None,
    "saved_thresholds": thresholds_loaded,
    "generated_files": sorted(
        [p.name for p in EVIDENCE_DIR.iterdir() if p.is_file()]
    ),
}

(EVIDENCE_DIR / "evidence_manifest.json").write_text(
    json.dumps(manifest, indent=2, default=str),
    encoding="utf-8"
)

print("=" * 90)
print("SECURESENSE RESEARCH EVIDENCE COLLECTION COMPLETE")
print("=" * 90)
print("Evidence directory:", EVIDENCE_DIR)
print("Generated files:", len(manifest["generated_files"]))
print("\nIMPORTANT:")
print("1. Do not retrain from this notebook.")
print("2. Review the threshold sweep and unseen-source recall carefully.")
print("3. Use only exported/verified values in the research paper.")
print("4. Keep primary stratified results separate from unseen-source results.")
